# 01 — Dataset Design + Hugging Face `datasets`

**Overview:** dataset design principles (splits, leakage, imbalance, train-only scaling) on a
small synthetic tabular dataset, then the same ideas via Hugging Face `datasets`
(`map`/`filter`/`select`/`shuffle`/`train_test_split`/batched/streaming) on a tiny real text
dataset (with a fully-synthetic, no-network fallback).

**Required libraries:** `numpy`, `pandas`, `scikit-learn`, `datasets`.

```
!pip install -q datasets   # not preinstalled on Colab
```

Connects to: `03_data_libraries` (`datasets` is Arrow-backed, same format as PyArrow). Leads to:
`05_ml_mathematics` and Day 2's ML fundamentals, which assume correct splits/no leakage.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **ML** | Machine Learning | Learning patterns from data to make predictions, instead of hand-coding rules. |

In [9]:
%pip install -q datasets

## 0. Environment

In [11]:
import sys
import numpy as np
import pandas as pd
print("Python:", sys.version.split()[0])
print("numpy:", np.__version__, "pandas:", pd.__version__)
SEED = 42
rng = np.random.default_rng(SEED)


Python: 3.13.16
numpy: 2.1.3 pandas: 2.2.3


## 1. Samples, features, labels — a synthetic dataset

💡 **Key Idea:** a sample = one row; features = its inputs; label = what we want to predict.

In [12]:
n = 500
feature_1 = rng.normal(loc=0, scale=1, size=n)
feature_2 = rng.normal(loc=5, scale=2, size=n)
noise = rng.normal(scale=0.5, size=n)
label = (feature_1 + 0.3 * feature_2 + noise > 3).astype(int)   # imbalanced on purpose

df = pd.DataFrame({"feature_1": feature_1, "feature_2": feature_2, "label": label})
print(df.head())
print("\nlabel balance:\n", df["label"].value_counts(normalize=True))


   feature_1  feature_2  label
0   0.304717   7.727724      0
1  -1.039984   6.790370      0
2   0.750451   3.561040      0
3   0.940565   1.994993      0
4  -1.951035  -0.929058      0

label balance:
 label
0    0.91
1    0.09
Name: proportion, dtype: float64


## 2. Imbalance

⚠️ **Common Mistake:** reporting accuracy on an imbalanced dataset without checking the
majority-class baseline first.

In [ ]:
majority_class_rate = df["label"].value_counts(normalize=True).max()
print(f"majority class baseline accuracy (always predict the majority class): {majority_class_rate:.3f}")
print("Any model scoring BELOW this baseline is worse than doing nothing.")
assert 0.5 <= majority_class_rate < 1.0


majority class baseline accuracy (always predict the majority class): 0.910
Any model scoring BELOW this baseline is worse than doing nothing.


## 3. Train/validation/test split — done correctly vs. with leakage

⚠️ **Common Mistake:** fitting a scaler on the FULL dataset before splitting lets validation/test
statistics leak into training.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["feature_1", "feature_2"]].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print("train size:", X_train.shape, "test size:", X_test.shape)
print("stratified -> similar label balance in both:",
      y_train.mean().round(3), "vs", y_test.mean().round(3))

# CORRECT: fit the scaler on train only, then transform both.
correct_scaler = StandardScaler().fit(X_train)
X_train_correct = correct_scaler.transform(X_train)
X_test_correct = correct_scaler.transform(X_test)

# LEAKY: fit the scaler on the full dataset BEFORE splitting.
leaky_scaler = StandardScaler().fit(X)
X_leaky_scaled = leaky_scaler.transform(X)
X_train_leaky, X_test_leaky, _, _ = train_test_split(X_leaky_scaled, y, test_size=0.2, random_state=SEED, stratify=y)

print("\ncorrect scaler mean (fit on train only):", correct_scaler.mean_.round(3))
print("leaky scaler mean   (fit on full data):   ", leaky_scaler.mean_.round(3))
print("\nThese differ because the leaky scaler already 'saw' the test distribution before any "
      "split happened -- in a real pipeline this inflates validation metrics optimistically.")


train size: (400, 2) test size: (100, 2)
stratified -> similar label balance in both: 0.09 vs 0.09

correct scaler mean (fit on train only): [0.01  4.923]
leaky scaler mean   (fit on full data):    [-0.013  4.911]

These differ because the leaky scaler already 'saw' the test distribution before any split happened -- in a real pipeline this inflates validation metrics optimistically.


## 4. Duplicates and leakage via row overlap

In [5]:
df_with_dupes = pd.concat([df, df.sample(20, random_state=SEED)], ignore_index=True)
print("rows with duplicates:", len(df_with_dupes), "vs original:", len(df))

# Splitting BEFORE deduplicating can place the same row in both train and test.
train_idx, test_idx = train_test_split(df_with_dupes.index, test_size=0.2, random_state=SEED)
overlap = set(df_with_dupes.loc[train_idx].duplicated(keep=False).index) & set(test_idx)
dup_rows = df_with_dupes[df_with_dupes.duplicated(keep=False)]
print("duplicate rows present in the dataset:", len(dup_rows))
print("Fix: df.drop_duplicates() BEFORE calling train_test_split, not after.")

deduped = df_with_dupes.drop_duplicates()
assert len(deduped) == len(df)


rows with duplicates: 520 vs original: 500
duplicate rows present in the dataset: 40
Fix: df.drop_duplicates() BEFORE calling train_test_split, not after.


## 5. Hugging Face `datasets`: Dataset / DatasetDict

🎯 **Interview Tip:** `datasets.Dataset` is Arrow-backed (see `03_pyarrow.ipynb`) — that's why
`.map(batched=True)` and streaming both scale well beyond what fits in memory.

In [6]:
from datasets import Dataset, DatasetDict

hf_dataset = Dataset.from_pandas(deduped.reset_index(drop=True))
print(hf_dataset)
print(hf_dataset[0])

assert len(hf_dataset) == len(deduped)


Dataset({
    features: ['feature_1', 'feature_2', 'label'],
    num_rows: 500
})
{'feature_1': 0.30471707975443135, 'feature_2': 7.727724459627819, 'label': 0}


## 6. map / filter / select / shuffle / train_test_split

In [7]:
def add_feature_sum(batch):
    return {"feature_sum": [a + b for a, b in zip(batch["feature_1"], batch["feature_2"])]}

hf_dataset = hf_dataset.map(add_feature_sum, batched=True)
print("after map, columns:", hf_dataset.column_names)
print(hf_dataset[0])

positives = hf_dataset.filter(lambda ex: ex["label"] == 1)
print("\nfiltered to label==1:", len(positives), "/", len(hf_dataset))

first_five = hf_dataset.select(range(5))
print("\nselected first 5 rows:", first_five["label"])

shuffled = hf_dataset.shuffle(seed=SEED)
print("\nshuffled first label (should usually differ from unshuffled):",
      shuffled[0]["label"], "vs original", hf_dataset[0]["label"])

split = hf_dataset.train_test_split(test_size=0.2, seed=SEED)
print("\ntrain_test_split:", split)
assert isinstance(split, DatasetDict)
assert len(split["train"]) + len(split["test"]) == len(hf_dataset)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

after map, columns: ['feature_1', 'feature_2', 'label', 'feature_sum']
{'feature_1': 0.30471707975443135, 'feature_2': 7.727724459627819, 'label': 0, 'feature_sum': 8.03244153938225}


Filter:   0%|          | 0/500 [00:00<?, ? examples/s]


filtered to label==1: 45 / 500

selected first 5 rows: Column([0, 0, 0, 0, 0])

shuffled first label (should usually differ from unshuffled): 0 vs original 0

train_test_split: DatasetDict({
    train: Dataset({
        features: ['feature_1', 'feature_2', 'label', 'feature_sum'],
        num_rows: 400
    })
    test: Dataset({
        features: ['feature_1', 'feature_2', 'label', 'feature_sum'],
        num_rows: 100
    })
})


## 7. remove_columns

In [8]:
trimmed = hf_dataset.remove_columns(["feature_sum"])
print("columns after remove_columns:", trimmed.column_names)
assert "feature_sum" not in trimmed.column_names
assert "feature_sum" in hf_dataset.column_names   # original untouched


columns after remove_columns: ['feature_1', 'feature_2', 'label']


## 8. A tiny real text dataset, with a synthetic fallback

🔬 **Experiment:** try loading a small Hugging Face Hub dataset; fall back to a fully synthetic
local one if there's no network access, so this notebook always runs.

In [9]:
try:
    from datasets import load_dataset
    text_ds = load_dataset("rotten_tomatoes", split="train[:200]")
    source = "huggingface_hub:rotten_tomatoes[:200]"
except Exception as e:
    print("Falling back to synthetic text dataset (no network / load failed):", repr(e))
    synthetic_texts = [
        {"text": f"synthetic review number {i} about the product", "label": i % 2}
        for i in range(200)
    ]
    text_ds = Dataset.from_list(synthetic_texts)
    source = "synthetic_fallback"

print("source:", source)
print(text_ds)
print(text_ds[0])
assert len(text_ds) == 200


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

source: huggingface_hub:rotten_tomatoes[:200]
Dataset({
    features: ['text', 'label'],
    num_rows: 200
})
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


## 9. Batched preprocessing on the text dataset

In [10]:
def lowercase_and_length(batch):
    lowered = [t.lower() for t in batch["text"]]
    return {"text": lowered, "text_length": [len(t) for t in lowered]}

text_ds = text_ds.map(lowercase_and_length, batched=True, batch_size=32)
print(text_ds[0])
assert all(c.islower() or not c.isalpha() for c in text_ds[0]["text"])


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1, 'text_length': 177}


## 10. Streaming (concept + demo)

💡 **Key Idea:** streaming mode never materializes the whole dataset — it's the dataset-level
analog of the `batched()` generator from `01_python`.

In [11]:
from datasets import Dataset as _Dataset

def example_generator():
    for i in range(10_000_000):   # pretend this is far too large to fit in memory
        yield {"idx": i, "value": i * i}

# IterableDataset.from_generator never materializes all 10M rows up front.
from datasets import IterableDataset
streaming_ds = IterableDataset.from_generator(example_generator)

first_three = list(streaming_ds.take(3))
print("first 3 streamed examples (out of a notional 10,000,000):", first_three)
assert len(first_three) == 3


first 3 streamed examples (out of a notional 10,000,000): [{'idx': 0, 'value': 0}, {'idx': 1, 'value': 1}, {'idx': 2, 'value': 4}]


## 11. Connection to what's next

The `feature_1`/`feature_2`/`label` synthetic dataset built here is exactly the shape `05_ml_mathematics`
and Day 2's regression/classification notebooks will reuse — correct splitting and scaling
matters just as much once a real model is involved.

## 12. Interview questions
1. Walk through why fitting a scaler on the full dataset before splitting causes leakage.
2. Why does `stratify=y` matter when splitting an imbalanced dataset?
3. What's the practical benefit of `.map(batched=True)` over row-by-row `.map()`?
4. When would `IterableDataset`/streaming be necessary instead of a regular `Dataset`?
5. Why must duplicates be removed before splitting, not after?

## 13. Exercises
1. Repeat the leaky-vs-correct scaling experiment with `MinMaxScaler` instead of `StandardScaler`.
2. Add a `.filter()` step that keeps only text examples longer than 20 characters, and report how many were dropped.
3. Modify `example_generator` to yield `{"idx": i, "is_even": i % 2 == 0}` and stream the first 5 even-indexed examples only using `.filter()` before `.take()`.

## 14. Summary

✅ Correct splitting (fit transforms on train only, dedupe before splitting, stratify on
imbalance) prevents the most common source of over-optimistic ML results. Hugging Face
`datasets` gives you `map`/`filter`/`select`/`shuffle`/`train_test_split`/streaming on top of the
same Arrow format from `03_data_libraries`. This closes Day 1's data section — next:
`05_ml_mathematics`.